# 📦 Freight Rate Extractor
Converts messy carrier Excel quote sheets into a clean, structured table.

### How to use:
1. Paste your Gemini API key in the cell below (get a free one at [aistudio.google.com](https://aistudio.google.com/apikey))
2. Click **Runtime → Run all**
3. When prompted, upload your Excel file(s)
4. The clean Excel file will download automatically

---
> **Free API key**: Go to https://aistudio.google.com/apikey → Sign in with Google → Create API key. It is free, no credit card needed.

In [ ]:
# ── STEP 1: Install packages ──────────────────────────────────────────────────
!pip install -q google-genai pydantic pandas openpyxl

In [ ]:
# ── STEP 2: Paste your free Gemini API key here ───────────────────────────────
GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"   # <-- replace this, keep the quotes

In [ ]:
# ── STEP 3: Run the extractor (no changes needed below this line) ─────────────
import os, json
import pandas as pd
from google.colab import files
from typing import List, Optional
from pydantic import BaseModel, Field
from google import genai
from google.genai import types


# ── Output schema ─────────────────────────────────────────────────────────────
class RateEntry(BaseModel):
    validity_from: str = Field(description="Start date of validity in YYYY-MM-DD format.")
    validity_to:   str = Field(description="End date of validity in YYYY-MM-DD format.")
    origin_city:   str = Field(description="Origin port or city name (e.g. Shanghai, Xiamen, Ningbo, Port Klang).")
    destination_city: str = Field(description="Destination city (e.g. Toronto).")
    currency:      str = Field(default="USD", description="Currency for all monetary values (USD or CAD).")

    # Ocean freight – base rates
    ocean_freight_20dc: Optional[float] = Field(None, description="Base ocean freight rate for 20DC or 20GP container.")
    ocean_freight_40dc: Optional[float] = Field(None, description="Base ocean freight rate for 40DC container, if listed.")
    ocean_freight_40hc: Optional[float] = Field(None, description="Base ocean freight rate for 40HC container.")

    # Destination / local charges – each has an amount AND a unit label
    handling_import_amount: Optional[float] = Field(None, description="Handling or Handover Fee (Import) amount.")
    handling_import_unit:   Optional[str]   = Field(None, description="Unit for handling fee: 'per shipment' or 'per container'.")

    destination_delivery_amount: Optional[float] = Field(None, description="Local delivery / cartage fee to the destination city (e.g. delivery to Toronto warehouse). This is usually charged per container.")
    destination_delivery_unit:   Optional[str]   = Field(None, description="Unit for delivery fee: 'per container' or 'per shipment'.")
    destination_delivery_note:   Optional[str]   = Field(None, description="Any special note about the delivery, e.g. 'live unload only', 'drop and pick'.")

    environment_fee_amount: Optional[float] = Field(None, description="Environmental fee or surcharge amount.")
    environment_fee_unit:   Optional[str]   = Field(None, description="Unit: 'per shipment' or 'per container'.")

    dsv_protect_amount: Optional[float] = Field(None, description="DSV Protect or cargo insurance fee amount.")
    dsv_protect_unit:   Optional[str]   = Field(None, description="Unit: 'per shipment' or 'per container'.")

    grade_a_handling_amount: Optional[float] = Field(None, description="Grade A container handling fee amount.")
    grade_a_handling_unit:   Optional[str]   = Field(None, description="Unit: 'per shipment' or 'per container'.")

    other_fees_amount: Optional[float] = Field(None, description="Total of any other local charges not listed above.")
    other_fees_note:   Optional[str]   = Field(None, description="Short description of what the other fees are.")

class RateSheetExtraction(BaseModel):
    rates: List[RateEntry] = Field(description="One entry per origin-destination lane.")


# ── Extraction function ───────────────────────────────────────────────────────
def extract_rates(csv_content: str, api_key: str) -> pd.DataFrame:
    client = genai.Client(api_key=api_key)

    system_instruction = """You are an expert logistics data analyst extracting structured freight rates from messy carrier quotation spreadsheets.

Extraction rules:
1. Create ONE row per origin city per destination city.
2. Destination / local charges (Handling, Delivery, Environmental, DSV Protect, Grade A handling, etc.)
   apply to ALL origin cities unless the document explicitly says otherwise. Copy them to every row.
3. For EVERY fee, capture both the AMOUNT and the UNIT ('per shipment', 'per container', 'per B/L', etc.).
   The unit label is critical — do not skip it.
4. The 'destination_delivery' fields are for the local delivery/cartage fee to the destination city
   (e.g. 'Delivery to Toronto'). This is typically charged per container.
5. Convert all dates to YYYY-MM-DD format.
6. If a container size rate is not listed, leave its field null.
7. Record the currency (USD or CAD).
"""

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Extract all freight rates from this carrier quotation data:\n\n{csv_content}",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            response_mime_type="application/json",
            response_schema=RateSheetExtraction,
            temperature=0.1
        ),
    )

    if not response.text:
        print("⚠️  Model returned an empty response.")
        return pd.DataFrame()

    data = json.loads(response.text)
    rates = data.get("rates", [])

    if not rates:
        print("⚠️  No rates found in the model response.")
        return pd.DataFrame()

    df = pd.DataFrame(rates)
    print(f"✅  Extracted {len(df)} lane(s) | Origins: {df['origin_city'].tolist()}")

    rename = {
        'validity_from':              'Validity From',
        'validity_to':                'Validity To',
        'origin_city':                'Origin City',
        'destination_city':           'Destination City',
        'currency':                   'Currency',
        'ocean_freight_20dc':         'Ocean Freight 20DC',
        'ocean_freight_40dc':         'Ocean Freight 40DC',
        'ocean_freight_40hc':         'Ocean Freight 40HC',
        'handling_import_amount':     'Handling Import (Amount)',
        'handling_import_unit':       'Handling Import (Unit)',
        'destination_delivery_amount':'Destination Delivery Fee (Amount)',
        'destination_delivery_unit':  'Destination Delivery Fee (Unit)',
        'destination_delivery_note':  'Delivery Note',
        'environment_fee_amount':     'Environment Fee (Amount)',
        'environment_fee_unit':       'Environment Fee (Unit)',
        'dsv_protect_amount':         'DSV Protect (Amount)',
        'dsv_protect_unit':           'DSV Protect (Unit)',
        'grade_a_handling_amount':    'Grade A Handling (Amount)',
        'grade_a_handling_unit':      'Grade A Handling (Unit)',
        'other_fees_amount':          'Other Fees (Amount)',
        'other_fees_note':            'Other Fees (Note)',
    }
    return df.rename(columns={k: v for k, v in rename.items() if k in df.columns})


# ── Upload & process ──────────────────────────────────────────────────────────
if GEMINI_API_KEY == "PASTE_YOUR_KEY_HERE":
    raise ValueError("❌ Please paste your Gemini API key in Step 2 above, then run again.")

print("📂 Please upload your carrier rate Excel or CSV file(s):")
uploaded = files.upload()

all_dfs = []

for file_name in uploaded.keys():
    print(f"\n{'─'*60}")
    print(f"Processing: {file_name}")
    print(f"{'─'*60}")

    with open(file_name, 'wb') as f:
        f.write(uploaded[file_name])

    try:
        if file_name.lower().endswith(('.xlsx', '.xls')):
            df_raw = pd.read_excel(file_name, header=None)
            print(f"Loaded Excel file ({df_raw.shape[0]} rows × {df_raw.shape[1]} cols)")
        else:
            df_raw = pd.read_csv(file_name, header=None)
            print(f"Loaded CSV file ({df_raw.shape[0]} rows × {df_raw.shape[1]} cols)")

        df_condensed = df_raw.dropna(how='all').dropna(how='all', axis=1)
        csv_text = df_condensed.to_csv(index=False, header=False)
        print(f"Compressed to {df_condensed.shape[0]} rows × {df_condensed.shape[1]} cols (empty rows/cols removed)")

    except Exception as e:
        print(f"❌ Could not read file: {e}")
        continue

    try:
        df_result = extract_rates(csv_text, GEMINI_API_KEY)
        if not df_result.empty:
            df_result.insert(0, 'Source File', file_name)
            all_dfs.append(df_result)
            print(f"🎉 Success!")
    except Exception as e:
        print(f"❌ Extraction failed: {e}")

# ── Save & download ───────────────────────────────────────────────────────────
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)

    output_name = "Extracted_Freight_Rates.xlsx"

    with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
        final_df.to_excel(writer, sheet_name='FCL Rates', index=False)

        # Auto-fit column widths
        ws = writer.sheets['FCL Rates']
        for col in ws.columns:
            max_len = max((len(str(cell.value)) if cell.value else 0) for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

    print(f"\n{'='*60}")
    print(f"✅  All done! {len(final_df)} total rows extracted.")
    print(f"Downloading: {output_name}")
    print(f"{'='*60}")
    files.download(output_name)
else:
    print("\n❌ No data was extracted. Please check:")
    print("   1. Your API key is correct (no extra spaces)")
    print("   2. The file uploaded correctly")
    print("   3. The file contains rate data (not empty)")